### Pydantic Schema's

In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
models = init_chat_model("groq:qwen/qwen3-32b")
models

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10cec8c80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10d009af0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel , Field
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The director of the movie")
    release_year: int = Field(description="The year the movie was released")

In [9]:
model_with_structured = models.with_structured_output(Movie)
response = model_with_structured.invoke("give me information about the movie 'Inception'?")
print(response)

title='Inception' director='Christopher Nolan' release_year=2010


In [10]:
class Actor(BaseModel):
    name : str = Field(description="The name of the actor")
    role : str = Field(description="The role of the actor in the movie")

class MovieDetails(BaseModel):
    title: str = Field(description="The title of the movie")
    cast: list[Actor] = Field(description="The cast of the movie")
    budget: float = Field(description="The budget of the movie in USD")


model_with_structured = models.with_structured_output(MovieDetails)
response = model_with_structured.invoke("give me detailed information about the movie 'Inception'?")
print(response)

title='Inception' cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role="Cobb's Reflection / Decoy"), Actor(name='Ken Watanabe', role='Professor Fujito')] budget=160000000.0


### How to see what are the Capabilities of the models which we are using ?

* use the models . profile

In [15]:
models.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DATA CLASS


In [17]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class MovieInfo:
    title: str
    director: str
    release_year: int   

agent = create_agent(models, response_format=MovieInfo)

agent_response = agent.invoke({"messages": [{"role": "user", "content": "give me information about the movie 'Inception'?"}]})
print(agent_response["structured_response"])

MovieInfo(title='Inception', director='Christopher Nolan', release_year=2010)


In [8]:
import re 
from typing import TypedDict

class SentenceResult(TypedDict):
    count: int
    matches: list

def find_sentence(sentence:str)->SentenceResult:
    pattern = r'Iphone\d+'
    matcher = re.findall(pattern, sentence)
    return {
        "count": len(matcher),
        "matches": matcher
    }

sentence = "Iphone10hasBigMarketIphone11hasdevelopedbyAppleIphone9isgoodModel"

result = find_sentence(sentence)
result

{'count': 3, 'matches': ['Iphone10', 'Iphone11', 'Iphone9']}